# AutoData — Adaptive feature-profile GPU validation

Validates the dataset profiler/selector on PaySim against manual E0/E1/E2 choices. The selector uses **only source structure** (repeat coverage, strict prior-history coverage, temporal availability); it never sees model scores when making its recommendation.


**Speed build:** shared preprocessing/feature-family cache + CUDA AMP + GPU tensor preloading are enabled for the benchmark.


In [ ]:
# 0) Confirm Colab GPU runtime
import os, sys, json, zipfile, copy
from pathlib import Path
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Colab: Runtime > Change runtime type > T4/L4/A100")
print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi || true


In [ ]:
# 1) Locate or upload the latest AutoData project ZIP
CONTENT = Path('/content')
PROJECT_ROOT = CONTENT / 'transaction-data-intelligence'

if not (PROJECT_ROOT / 'src').exists():
    candidates = list(CONTENT.glob('AutoData*.zip')) + list(CONTENT.glob('*.zip'))
    zpath = candidates[0] if candidates else None
    if zpath is None:
        from google.colab import files
        print('Upload AutoData_v2_adaptive_features.zip')
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No project ZIP uploaded')
        zpath = CONTENT / next(iter(uploaded))
    print('Extracting', zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(CONTENT)

if not (PROJECT_ROOT / 'src').exists():
    matches = [p.parent for p in CONTENT.rglob('config.yaml') if (p.parent / 'src').exists()]
    if len(matches) != 1:
        raise RuntimeError(f'Could not uniquely locate project root: {matches}')
    PROJECT_ROOT = matches[0]

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
# 2) Install core backend dependencies
!pip install -q -r requirements.txt
import torch
assert torch.cuda.is_available()
DEVICE='cuda'
print('GPU ready:', torch.cuda.get_device_name(0))


## Dataset setup

Download the PaySim CSV separately and place it in Google Drive. The commonly used filename is similar to `PS_20174392719_1491204439457_log.csv`.

The default source cap keeps feature generation T4/Colab-RAM friendly while preserving a continuous chronological slice. Increase only after the first run succeeds.


In [ ]:
# 3) Controls — larger T4 validation for the adaptive selector
from google.colab import drive
drive.mount('/content/drive')

PAYSIM_PATH = '/content/drive/MyDrive/paysim/PS_20174392719_1491204439457_log.csv'
PROFILE = 'strict'
MAX_SOURCE_ROWS = 4_000_000   # continuous chronological source slice; lower if Colab RAM is constrained
ROWS = 1_000_000              # controlled benchmark rows from that source slice
SEEDS = [42, 123, 456]       # core E0/E1/E2/AUTO evidence
ABLATION_SEEDS = [42]          # diagnostic feature-family runs; set =SEEDS for full 3-seed ablations
EPOCHS = 10
PATIENCE = 4
BATCH_SIZE = 512              # T4-safe for the small sanity transformer; lower to 256 only on OOM

print({
    'PROFILE': PROFILE, 'MAX_SOURCE_ROWS': MAX_SOURCE_ROWS, 'ROWS': ROWS,
    'SEEDS': SEEDS, 'ABLATION_SEEDS': ABLATION_SEEDS, 'EPOCHS': EPOCHS, 'BATCH_SIZE': BATCH_SIZE,
})


In [ ]:
# 4) Load PaySim and apply the explicit schema/time adapter
import pandas as pd
from src.ingestion.adapters import adapt_paysim
from src.ingestion.schema_detector import detect_schema
from src.ingestion.roles import detect_roles, schema_for_profiling
from src.preprocessing.levels import infer_roles
from src.utils.config import load_config

path = Path(PAYSIM_PATH)
if not path.exists():
    raise FileNotFoundError(f'PaySim CSV not found: {path}')

raw = pd.read_csv(path, nrows=MAX_SOURCE_ROWS)
print(f'Loaded source rows: {len(raw):,}')
print('Source frauds:', int(pd.to_numeric(raw['isFraud'], errors='coerce').fillna(0).sum()))
print('Source fraud rate:', float(pd.to_numeric(raw['isFraud'], errors='coerce').mean()))

adapter = adapt_paysim(raw, profile=PROFILE)
df = adapter.df
schema = detect_schema(df, f'paysim_{PROFILE}')
roles = detect_roles(df, schema, target=adapter.target).with_overrides(
    df,
    role_overrides=adapter.role_overrides,
    entity=adapter.entity,
    datetime=adapter.datetime,
)
ps = schema_for_profiling(schema, roles, df)
level_roles = infer_roles(df, ps, roles, hints=adapter.hints)

print('Adapter profile:', adapter.profile)
print('Dropped:', adapter.dropped_columns)
print('Dataset roles:', {'target': roles.target, 'entity': roles.entity, 'datetime': roles.datetime, 'task': roles.task})
print('E2 roles:', level_roles.as_dict())
print('Notes:')
for n in adapter.notes:
    print('-', n)

assert roles.task == 'binary_classification'
assert level_roles.amount == 'amount'
assert level_roles.category == 'type'
assert level_roles.merchant == 'nameDest'


In [ ]:
# 5) Profile structural support and obtain an auditable recommendation
from src.profiling.feature_profile import recommend_for_autodata, apply_feature_recommendation

cfg = load_config()
recommendation = recommend_for_autodata(df, level_roles, cfg)
print(json.dumps(recommendation.to_dict(), indent=2, default=str))

recommended_cfg = apply_feature_recommendation(cfg, recommendation)
print('AUTO RECOMMENDATION:', {
    'level': recommendation.level,
    'numeric_mode': recommendation.numeric_mode,
    'feature_groups': list(recommendation.feature_groups),
    'confidence': recommendation.confidence,
})


In [ ]:
# 6) Natural validation/test; fraud-enriched training only
from src.preprocessing.levels import DataPreparer

benchmark_cfg = copy.deepcopy(cfg)
benchmark_cfg['sampling']['train_positive_share'] = 0.10
benchmark_cfg['sampling']['validation_positive_share'] = None
benchmark_cfg['sampling']['keep_all_test_positives'] = False

shared_preparation_cache = {}
preparer = DataPreparer(df, ps, level_roles, benchmark_cfg, rows=ROWS, seed=cfg['project']['seed'], shared_cache=shared_preparation_cache)
split_info = preparer.prepare_split()
print(json.dumps(split_info['sample'], indent=2, default=str))
for split in ['train','validation','test']:
    s=split_info['sample'][split]
    print(split, 'rows=', s['rows'], 'positives=', s['positives'], 'sample prevalence=', s['sample_positive_share'])
if split_info['sample']['test']['positives'] < 100:
    print('CAUTION: fewer than 100 natural fraud positives in test; use the result directionally and consider a larger/full holdout.')


In [ ]:
# 7) Build the recommended profile and require point-in-time safety whenever E2 is selected
probe_cfg = copy.deepcopy(benchmark_cfg)
probe_cfg = apply_feature_recommendation(probe_cfg, recommendation)
probe = DataPreparer(df, ps, level_roles, probe_cfg, rows=ROWS, seed=cfg['project']['seed'], shared_cache=shared_preparation_cache)
probe.prepare_split()
prepared = probe.build(recommendation.level)
if recommendation.level == 'E2':
    pit = prepared.info.get('point_in_time', {})
    print(json.dumps(pit, indent=2, default=str))
    assert pit.get('passed'), pit
    print('PASS: recommended behavioral features are point-in-time safe.')
else:
    print('Selector chose E1; behavioral point-in-time features are not part of the recommendation.')


In [ ]:
# 8) Compare automatic recommendation against fixed manual baselines/ablations
import pandas as pd
from src.evaluation.experiment import run_comparison_multiseed, aggregate_seeds

settings={'epochs':EPOCHS,'patience':PATIENCE,'batch_size':BATCH_SIZE,'amp':True,'preload_to_device':True}

def make_preparer(config):
    p=DataPreparer(df, ps, level_roles, config, rows=ROWS, seed=cfg['project']['seed'], shared_cache=shared_preparation_cache)
    p.prepare_split()
    return p

def run_variant(name, level, config, seeds=None):
    seeds = list(seeds or SEEDS)
    print(f'\n=== {name} ===  seeds={seeds}')
    p=make_preparer(config)
    if level=='E2':
        b=p.build('E2')
        enabled_groups=set(config['features'].get('enabled_groups', []))
        needs_history_audit=bool({'history','sequence'} & enabled_groups)
        if needs_history_audit:
            pit=b.info.get('point_in_time',{})
            assert pit.get('passed'), f"Point-in-time audit failed for {name}: {pit}"
    out=run_comparison_multiseed(
        p,[level],config,seeds,settings=settings,device='cuda',
        progress=lambda done,total,seed,r: print(
            f"  seed={seed}: PR-AUC={r.metrics['test_unweighted']['pr_auc']:.6f} "
            f"F1={r.metrics['test_unweighted']['f1']:.6f}")
    )
    table=aggregate_seeds(out).reset_index().rename(columns={'Level':'level'})
    table.insert(0,'variant',name)
    rows=[]
    for seed, results in out.items():
        m=results[0].metrics['test_unweighted']
        rows.append({'seed':seed, **{k:m[k] for k in ['pr_auc','roc_auc','precision','recall','f1']}})
    u=pd.DataFrame(rows)
    for metric in ['pr_auc','roc_auc','precision','recall','f1']:
        table[f'natural_{metric} mean']=u[metric].mean()
        table[f'natural_{metric} std']=u[metric].std(ddof=1)
    return table

print('GPU fast path: AMP=True, preload_to_device=True')
print('Shared preprocessing cache enabled across all variants.')
print(f'Core seeds={SEEDS}; diagnostic ablation seeds={ABLATION_SEEDS}; batch_size={BATCH_SIZE}')
variants=[]
# Raw baseline
c=copy.deepcopy(benchmark_cfg)
variants.append(run_variant('E0_raw','E0',c))
# Conservative continuous E1
c=copy.deepcopy(benchmark_cfg)
c['representation']['numeric_mode']='continuous'
c['representation']['numeric_coarse_bins']=0
variants.append(run_variant('E1_continuous','E1',c))
# Full E2 validated on the first dataset
c=copy.deepcopy(benchmark_cfg)
c['features']['enabled_groups']=['temporal','history','sequence']
variants.append(run_variant('E2_full','E2',c))
# Feature-family ablations that explain *why* a schema differs
for name, groups in [
    ('E2_temporal',['temporal']),
    ('E2_history',['history']),
    ('E2_sequence',['sequence']),
    ('E2_temporal_history',['temporal','history']),
]:
    c=copy.deepcopy(benchmark_cfg)
    c['features']['enabled_groups']=groups
    variants.append(run_variant(name,'E2',c,seeds=ABLATION_SEEDS))
# Automatic structural recommendation. Avoid duplicate execution if it exactly equals E1 continuous.
auto_name='AUTO_' + recommendation.level + ('_' + '_'.join(recommendation.feature_groups) if recommendation.feature_groups else '_continuous')
auto_cfg=apply_feature_recommendation(copy.deepcopy(benchmark_cfg), recommendation)
manual_duplicate = recommendation.level == 'E1' and recommendation.numeric_mode == 'continuous'
if manual_duplicate:
    print('AUTO recommendation is exactly E1_continuous; no duplicate GPU run needed.')
else:
    variants.append(run_variant(auto_name,recommendation.level,auto_cfg))

result_table=pd.concat(variants,ignore_index=True)
display(result_table.sort_values('natural_pr_auc mean',ascending=False))


In [ ]:
# 9) Did the structural selector make a defensible choice?
pr_col='natural_pr_auc mean'
f1_col='natural_f1 mean'
view=result_table[['variant',pr_col,f1_col]].sort_values(pr_col,ascending=False)
display(view)

selected_variant = 'E1_continuous' if (recommendation.level=='E1' and recommendation.numeric_mode=='continuous') else 'AUTO_' + recommendation.level + ('_' + '_'.join(recommendation.feature_groups) if recommendation.feature_groups else '_continuous')
selected=view[view['variant']==selected_variant]
if len(selected):
    best_pr=float(view[pr_col].max())
    auto_pr=float(selected.iloc[0][pr_col])
    print('Selector PR-AUC gap to best tested profile:', best_pr-auto_pr)
    print('NOTE: the selector is intentionally conservative and never used these model scores to choose.')


In [ ]:
# 10) Export benchmark + adaptive-profile context
from datetime import datetime
from google.colab import files
stamp=datetime.now().strftime('%Y%m%d_%H%M%S')
result_path=f'adaptive_paysim_{stamp}.csv'
context_path=f'adaptive_paysim_context_{stamp}.json'
result_table.to_csv(result_path,index=False)
context={
    'dataset':'PaySim', 'profile':PROFILE,
    'source_rows_loaded':len(raw), 'rows_requested':ROWS,
    'source_frauds':int(pd.to_numeric(raw['isFraud'],errors='coerce').fillna(0).sum()),
    'source_fraud_rate':float(pd.to_numeric(raw['isFraud'],errors='coerce').mean()),
    'adaptive_recommendation':recommendation.to_dict(),
    'adapter':{
        'target':adapter.target,'entity':adapter.entity,'datetime':adapter.datetime,
        'hints':adapter.hints,'dropped_columns':list(adapter.dropped_columns),
        'notes':list(adapter.notes),'metadata':adapter.metadata,
    },
    'split':split_info,
    'seeds':SEEDS,'ablation_seeds':ABLATION_SEEDS,'epochs':EPOCHS,'patience':PATIENCE,'batch_size':BATCH_SIZE,
    'gpu_fast_path':{'amp':True,'preload_to_device':True,'shared_preparation_cache':True},
    'device':torch.cuda.get_device_name(0),
    'evaluation_distribution':'fraud-enriched train; natural temporal validation/test',
    'selector_policy':'source-structure only; model scores are evaluation evidence, never selector inputs',
}
Path(context_path).write_text(json.dumps(context,indent=2,default=str))
print(result_path,context_path)
files.download(result_path)
files.download(context_path)
